# 데이터 정규화 (Data Normalization)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import MinMaxScaler, StandardScaler

np.random.seed(42)
torch.manual_seed(42)

## 1. 데이터 정규화가 필요한 이유

모델은 입력값을 그대로 받아 연산한다.  
따라서 입력 특성들의 값 범위가 매우 다르면, 특정 특성이 지나치게 크게 작용할 수 있다.

- 나이: 20 ~ 60
- 연봉: 30000000 ~ 120000000
- 시험 점수: 0 ~ 100

이때 연봉은 숫자 크기 자체가 매우 크므로, 같은 가중치 변화라도 다른 특성보다 훨씬 큰 영향을 줄 수 있다.

즉 정규화는  
“특성들의 상대적인 중요도를 완전히 같게 만든다”기보다,  “숫자 크기 차이 때문에 학습이 불리해지는 상황을 줄여주는 작업”이다.

In [2]:
# 서로 스케일이 많이 다른 두 특성 예시
sample_df = pd.DataFrame({
    'study_hour': [1, 2, 3, 4, 5],
    'annual_income': [28000000, 35000000, 52000000, 80000000, 110000000]
})

print(sample_df)
print()
print(sample_df.describe())

   study_hour  annual_income
0           1       28000000
1           2       35000000
2           3       52000000
3           4       80000000
4           5      110000000

       study_hour  annual_income
count    5.000000   5.000000e+00
mean     3.000000   6.100000e+07
std      1.581139   3.394113e+07
min      1.000000   2.800000e+07
25%      2.000000   3.500000e+07
50%      3.000000   5.200000e+07
75%      4.000000   8.000000e+07
max      5.000000   1.100000e+08


## 2. Min-Max Scaling

Min-Max Scaling은 값을 0과 1 사이 범위로 맞추는 방식이다.

공식은 아래와 같다.

$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

이 방식은
- 최솟값은 0
- 최댓값은 1
- 나머지 값은 그 사이로 비율에 맞춰 변환된다.

즉 원래 값의 상대적 순서를 유지하면서 범위를 맞춰주는 방식이다.


In [3]:
minmax_scaler = MinMaxScaler()
sample_minmax = minmax_scaler.fit_transform(sample_df)

sample_minmax_df = pd.DataFrame(sample_minmax, columns=sample_df.columns)
print(sample_minmax_df)

   study_hour  annual_income
0        0.00       0.000000
1        0.25       0.085366
2        0.50       0.292683
3        0.75       0.634146
4        1.00       1.000000


## 3. Standardization

Standardization은 평균을 0, 표준편차를 1에 가깝게 맞추는 방식이다.

공식은 아래와 같다.

$$
x' = \frac{x - \mu}{\sigma}
$$

여기서
- $\mu$는 평균
- $\sigma$는 표준편차이다.

즉 각 값이 “평균으로부터 얼마나 떨어져 있는지”를 기준으로 다시 표현하는 방식이라고 볼 수 있다.


In [4]:
standard_scaler = StandardScaler()
sample_standard = standard_scaler.fit_transform(sample_df)

sample_standard_df = pd.DataFrame(sample_standard, columns=sample_df.columns)
print(sample_standard_df)
print()
print('변환 후 평균(대략 0):')
print(sample_standard_df.mean())
print()
print('변환 후 표준편차(대략 1):')
print(sample_standard_df.std(ddof=0))

   study_hour  annual_income
0   -1.414214      -1.087033
1   -0.707107      -0.856450
2    0.000000      -0.296464
3    0.707107       0.625867
4    1.414214       1.614079

변환 후 평균(대략 0):
study_hour       0.000000e+00
annual_income   -4.440892e-17
dtype: float64

변환 후 표준편차(대략 1):
study_hour       1.0
annual_income    1.0
dtype: float64


## 4. train 데이터 기준으로 정규화

실제 학습에서는 보통 먼저 train / test를 나누고, 그 다음 train 데이터 기준으로 scaler를 학습시켜야 한다.

test 데이터는 “모델이 처음 보는 데이터”처럼 다루어야 한다.  
그런데 정규화할 때 test 데이터의 평균이나 최댓값까지 미리 써버리면, 학습 단계에서 test 정보가 새어 들어간다.

이런 문제를 데이터 누수(data leakage)라고 한다.

In [5]:
# 간단한 train / test 예시
train_df = pd.DataFrame({
    'x1': [10, 20, 30, 40, 50],
    'x2': [1000, 1500, 2000, 2500, 3000]
})

test_df = pd.DataFrame({
    'x1': [60, 70],
    'x2': [3500, 4500]
})

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df)   # train으로 기준 학습
test_scaled = scaler.transform(test_df)         # test는 같은 기준으로 변환

print('train_scaled')
print(pd.DataFrame(train_scaled, columns=train_df.columns))
print()
print('test_scaled')
print(pd.DataFrame(test_scaled, columns=test_df.columns))


train_scaled
         x1        x2
0 -1.414214 -1.414214
1 -0.707107 -0.707107
2  0.000000  0.000000
3  0.707107  0.707107
4  1.414214  1.414214

test_scaled
         x1        x2
0  2.121320  2.121320
1  2.828427  3.535534


## 5. 정규화가 학습에 미치는 영향

정규화하면 학습 과정이 더 안정적이거나 빠르게 진행될 수 있다.

In [9]:
# 이진 분류 예제를 위한 toy 데이터 생성
n = 400     # 샘플 개수
x1 = np.random.uniform(0, 1, size=(n, 1))       # 0~1 사이의 값
x2 = np.random.uniform(0, 1000, size=(n, 1))    # 0~1000 사이의 값
X_raw = np.hstack([x1, x2])                     # 입력 데이터 (n, 2)

# 선형 결합 값
logit = 3 * x1.squeeze() + 0.005 * x2.squeeze() - 3

# 타깃 생성(이진 분류)
y = (logit > 0).astype(np.float32).reshape(-1, 1)

# 표준화 버전
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_raw_t = torch.tensor(X_raw, dtype=torch.float32)
X_scaled_t = torch.tensor(X_scaled, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

In [10]:
# 모델 정의/학습 함수 정의
class SimpleBinaryNet(nn.Module):
    def __init__(self):
        # 부모 클래스(nn.Module) 초기화
        # 파이토치에서 사용자 정의 모델을 만들 때 기본적으로 작성
        super().__init__()
        
        # 실제 모델 구조 정의
        self.net = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
        
    def forward(self, x):
        # 입력이 들어 왔을 때 어떤 계산을 할지 정의하는 함수
        # self.net에 x를 전달하며 순전파를 수행한다
        return self.net(x)
    
# 학습 함수
def train_model(X, y, epochs=100):
    model = SimpleBinaryNet()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    
    loss_history = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        loss_history.append(loss.item())
        
    return model, loss_history


raw_model, raw_loss_history = train_model(X_raw_t, y_t)
scaled_model, scaled_loss_history = train_model(X_scaled_t, y_t)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(raw_loss_history, label='raw input')
plt.plot(scaled_loss_history, label='scaled input')
plt.xlabel('Epoch')
plt.ylabel()

plt.legend()
plt.grid()
plt.show()